# Neuronale Netze mit NumPy: ein einfacher Einstieg

Dieses Notebook ist eine **komplett überarbeitete** und **deutschsprachige** Einführung in kleine neuronale Netze mit NumPy.

Der Fokus liegt auf drei Punkten:

1. **Verstehen**, wie ein kleines neuronales Netz aufgebaut ist.
2. **Korrekt implementieren**, wie Gradienten mit **automatischer Differenzierung** berechnet werden.
3. Das Ganze an einem einfachen Beispiel trainieren: Wir approximieren die Funktion $\sin(x)$.

Wir arbeiten bewusst **klein, langsam und transparent**.  
Das Ziel ist **nicht** maximale Geschwindigkeit, sondern ein einfacher Einstieg für Studierende.

## Was Sie nach diesem Notebook können sollten

Nach dem Durcharbeiten sollten Sie erklären können,

- was ein Parameter in einem neuronalen Netz ist,
- was bei einem Vorwärtsdurchlauf passiert,
- warum man für das Training Ableitungen braucht,
- wie die **Kettenregel** in einem Rechengraphen benutzt wird,
- und wie ein einfaches Netz mit Gradientenverfahren trainiert wird.

Wichtig: Unsere Implementierung ist nur zum Lernen der Konzepte.  
Bibliotheken wie PyTorch oder JAX sind deutlich leistungsfähiger. Hier bauen wir die Kernideen selbst nach.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Das Lernbeispiel

Wir wollen aus Daten die Funktion $\sin(x)$ lernen.

Warum ist das ein gutes Beispiel?

- Die Funktion ist bekannt.
- Man sieht sofort, ob die Vorhersage sinnvoll ist.
- Das Ziel ist leicht verständlich: Das Netz soll eine Kurve nachzeichnen.

In [ ]:
x_train = np.linspace(-np.pi, np.pi, 256).reshape(-1, 1)
y_train = np.sin(x_train)

dx = x_train[1, 0] - x_train[0, 0]
x_val = (np.linspace(-np.pi, np.pi, 256) + 0.5 * dx).reshape(-1, 1)
y_val = np.sin(x_val)

plt.figure(figsize=(7, 4))
plt.plot(x_train, y_train, label="Trainingsdaten: sin(x)")
plt.scatter(x_val[::8], y_val[::8], s=12, label="Validierungsdaten", alpha=0.7)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Datensatz")
plt.legend()
plt.show()

## Idee der automatischen Differenzierung

Wir bauen eine kleine Klasse `Tensor`.

Jedes Objekt speichert

- den **numerischen Wert** `data`,
- den **Gradienten** `grad`,
- und Informationen darüber, **aus welchen vorherigen Objekten** es entstanden ist.

Wenn wir am Ende einen skalaren Fehlerwert `loss` haben, rufen wir `loss.backward()` auf.

Dann passiert Folgendes:

1. Wir laufen den Rechengraphen **rückwärts** durch.
2. An jedem Knoten wenden wir die **lokale Ableitungsregel** an.
3. Die Gradienten werden mit der **Kettenregel** zusammengesetzt.

Genau das ist automatische Differenzierung.

## Eine kleine technische Hilfsfunktion

Bei Operationen wie

$$
XW + b
$$

wird der Bias $b$ über viele Zeilen hinweg **broadcastet**.  
Im Rückwärtslauf muss der Gradient daher wieder auf die richtige Form zurückgeführt werden.

Diese Hilfsfunktion erledigt genau das.

In [ ]:
def sum_to_shape(grad, shape):
    """Summiert einen Gradienten so, dass seine Form wieder zu `shape` passt.

    Das ist wichtig für Broadcasting, zum Beispiel bei `X @ W + b`.
    """
    grad = np.asarray(grad, dtype=float)

    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)

    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)

    return grad

## Die Klasse `Tensor`

Diese Klasse ist das Herzstück des Notebooks.

Wir implementieren nur wenige Operationen:

- Addition
- Subtraktion
- elementweise Multiplikation
- Matrixmultiplikation
- Potenzen
- Summe und Mittelwert
- Aktivierungsfunktionen `relu` und `tanh`

Das reicht bereits für ein kleines neuronales Netz.

In [ ]:
class Tensor:
    def __init__(self, data, requires_grad=False, _children=(), _op=""):
        self.data = np.array(data, dtype=float)
        self.requires_grad = requires_grad
        self.grad = np.zeros_like(self.data) if self.requires_grad else None

        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f"Tensor(data={self.data}, grad={self.grad})"

    def zero_grad(self):
        if self.requires_grad:
            self.grad = np.zeros_like(self.data)

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(
            self.data + other.data,
            requires_grad=self.requires_grad or other.requires_grad,
            _children=(self, other),
            _op="+",
        )

        def _backward():
            if self.requires_grad:
                self.grad += sum_to_shape(out.grad, self.data.shape)
            if other.requires_grad:
                other.grad += sum_to_shape(out.grad, other.data.shape)

        out._backward = _backward
        return out

    __radd__ = __add__

    def __neg__(self):
        out = Tensor(-self.data, requires_grad=self.requires_grad, _children=(self,), _op="neg")

        def _backward():
            if self.requires_grad:
                self.grad -= out.grad

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self + (-other)

    def __rsub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return other + (-self)

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(
            self.data * other.data,
            requires_grad=self.requires_grad or other.requires_grad,
            _children=(self, other),
            _op="*",
        )

        def _backward():
            if self.requires_grad:
                self.grad += sum_to_shape(out.grad * other.data, self.data.shape)
            if other.requires_grad:
                other.grad += sum_to_shape(out.grad * self.data, other.data.shape)

        out._backward = _backward
        return out

    __rmul__ = __mul__

    def pow(self, exponent):
        out = Tensor(
            self.data ** exponent,
            requires_grad=self.requires_grad,
            _children=(self,),
            _op=f"pow({exponent})",
        )

        def _backward():
            if self.requires_grad:
                self.grad += out.grad * exponent * (self.data ** (exponent - 1))

        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self * other.pow(-1)

    def __rtruediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return other / self

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(
            self.data @ other.data,
            requires_grad=self.requires_grad or other.requires_grad,
            _children=(self, other),
            _op="@",
        )

        def _backward():
            if self.requires_grad:
                self.grad += out.grad @ other.data.T
            if other.requires_grad:
                other.grad += self.data.T @ out.grad

        out._backward = _backward
        return out

    @property
    def T(self):
        out = Tensor(self.data.T, requires_grad=self.requires_grad, _children=(self,), _op="transpose")

        def _backward():
            if self.requires_grad:
                self.grad += out.grad.T

        out._backward = _backward
        return out

    def sum(self, axis=None, keepdims=False):
        out = Tensor(
            self.data.sum(axis=axis, keepdims=keepdims),
            requires_grad=self.requires_grad,
            _children=(self,),
            _op="sum",
        )

        def _backward():
            if not self.requires_grad:
                return

            grad = out.grad

            if axis is None:
                self.grad += np.ones_like(self.data) * grad
                return

            axes = axis if isinstance(axis, tuple) else (axis,)
            axes = tuple(a if a >= 0 else a + self.data.ndim for a in axes)

            if not keepdims:
                for ax in sorted(axes):
                    grad = np.expand_dims(grad, axis=ax)

            self.grad += np.ones_like(self.data) * grad

        out._backward = _backward
        return out

    def mean(self, axis=None, keepdims=False):
        if axis is None:
            denom = self.data.size
        else:
            axes = axis if isinstance(axis, tuple) else (axis,)
            denom = np.prod([self.data.shape[a] for a in axes])

        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / denom)

    def relu(self):
        out = Tensor(
            np.maximum(0.0, self.data),
            requires_grad=self.requires_grad,
            _children=(self,),
            _op="relu",
        )

        def _backward():
            if self.requires_grad:
                self.grad += out.grad * (self.data > 0.0)

        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Tensor(t, requires_grad=self.requires_grad, _children=(self,), _op="tanh")

        def _backward():
            if self.requires_grad:
                self.grad += out.grad * (1.0 - t ** 2)

        out._backward = _backward
        return out

    def backward(self):
        if self.data.size != 1:
            raise ValueError("`backward()` ist hier nur für skalare Ausgaben implementiert.")

        topo = []
        visited = set()

        def build(node):
            if node not in visited:
                visited.add(node)
                for parent in node._prev:
                    build(parent)
                topo.append(node)

        build(self)

        self.grad = np.ones_like(self.data)

        for node in reversed(topo):
            node._backward()

## Ein erster sehr kleiner Test

Wir prüfen die Ableitung von

$$
f(x) = x^2 + 3x + 1.
$$

Von Hand gilt

$$
f'(x) = 2x + 3.
$$

An der Stelle $x=2$ muss also $f'(2)=7$ herauskommen.

In [ ]:
x = Tensor([2.0], requires_grad=True)
f = x.pow(2) + 3.0 * x + 1.0
f.backward()

print("f(x) =", f.data)
print("berechneter Gradient =", x.grad)
print("erwarteter Gradient   =", 2 * 2.0 + 3.0)

Wenn hier $7$ herauskommt, ist das ein gutes erstes Signal.  
Aber für ein neuronales Netz reicht ein einzelner Test noch nicht.

Darum machen wir jetzt etwas Strengeres: einen **Gradientencheck** mit finiten Differenzen.

## Bausteine für ein kleines neuronales Netz

Wir definieren nun einfache Schichten und Container:

- `Linear`: affine Abbildung $x \mapsto xW + b$
- `Tanh` und `ReLU`: Aktivierungsfunktionen
- `Sequential`: führt mehrere Schichten nacheinander aus

Für dieses Notebook benutzen wir hauptsächlich `Tanh`, weil unsere Zielfunktion $\sin(x)$ sowohl positive als auch negative Werte annimmt.

In [ ]:
class Module:
    def parameters(self):
        return []

    def zero_grad(self):
        for p in self.parameters():
            p.zero_grad()


class Linear(Module):
    def __init__(self, in_features, out_features):
        # Xavier-Initialisierung: gut geeignet für tanh-Netze
        limit = np.sqrt(6.0 / (in_features + out_features))
        self.weight = Tensor(
            np.random.uniform(-limit, limit, size=(in_features, out_features)),
            requires_grad=True,
        )
        self.bias = Tensor(np.zeros((1, out_features)), requires_grad=True)

    def __call__(self, x):
        return x @ self.weight + self.bias

    def parameters(self):
        return [self.weight, self.bias]


class Tanh(Module):
    def __call__(self, x):
        return x.tanh()


class ReLU(Module):
    def __call__(self, x):
        return x.relu()


class Sequential(Module):
    def __init__(self, *layers):
        self.layers = list(layers)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            if isinstance(layer, Module):
                params.extend(layer.parameters())
        return params

## Verlustfunktion und Optimierer

Für die Regression nehmen wir den mittleren quadratischen Fehler

$$
\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^N (\hat y_i - y_i)^2.
$$

Außerdem verwenden wir **stochastischen Gradientenabstieg** (SGD).

Das ist nicht der modernste Optimierer, aber didaktisch sehr gut:
Man sieht direkt, dass jeder Parameterschritt einfach in Richtung des negativen Gradienten geht.

In [ ]:
def mse_loss(prediction, target):
    return ((prediction - target).pow(2)).mean()


class SGD:
    def __init__(self, parameters, lr=0.01):
        self.parameters = list(parameters)
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.zero_grad()

    def step(self):
        for p in self.parameters:
            p.data -= self.lr * p.grad

## Gradiententest mit finiten Differenzen

Bevor wir trainieren, prüfen wir die Korrektheit unserer Gradienten numerisch.

Idee:

1. Wir berechnen den Gradienten mit `backward()`.
2. Wir stören einen Parameter $w$ leicht nach oben und unten.
3. Wir approximieren die Ableitung mit

$$
\frac{L(w+\varepsilon)-L(w-\varepsilon)}{2\varepsilon}.
$$

Wenn beide Werte fast gleich sind, ist das ein starkes Indiz dafür, dass die Implementierung korrekt ist.

In [ ]:
np.random.seed(1)

test_layer = Linear(2, 1)
x_test = Tensor([[1.0, -2.0]])
y_test = Tensor([[0.5]])

loss = mse_loss(test_layer(x_test), y_test)
test_layer.zero_grad()
loss.backward()

autograd_grad = test_layer.weight.grad.copy()

epsilon = 1e-6
finite_diff_grad = np.zeros_like(test_layer.weight.data)

for i in range(test_layer.weight.data.shape[0]):
    for j in range(test_layer.weight.data.shape[1]):
        old_value = test_layer.weight.data[i, j]

        test_layer.weight.data[i, j] = old_value + epsilon
        loss_plus = mse_loss(test_layer(x_test), y_test).data.item()

        test_layer.weight.data[i, j] = old_value - epsilon
        loss_minus = mse_loss(test_layer(x_test), y_test).data.item()

        finite_diff_grad[i, j] = (loss_plus - loss_minus) / (2 * epsilon)
        test_layer.weight.data[i, j] = old_value

print("Gradient aus backward():")
print(autograd_grad)

print("\nGradient aus finiten Differenzen:")
print(finite_diff_grad)

print("\nMaximaler absoluter Fehler:")
print(np.max(np.abs(autograd_grad - finite_diff_grad)))

Ein sehr kleiner Fehler ist hier genau das, was wir sehen wollen.

Damit haben wir eine wesentlich bessere Grundlage als bei einer bloß optisch funktionierenden Trainingskurve.

## Das eigentliche Netz

Wir wählen ein kleines voll verbundenes Netz:

- Eingabegröße: 1
- zwei versteckte Schichten mit je 32 Neuronen
- `tanh` als Aktivierung
- Ausgabegröße: 1

Das ist klein genug zum Verstehen und groß genug, um $\sin(x)$ gut zu approximieren.

In [ ]:
np.random.seed(42)

model = Sequential(
    Linear(1, 32),
    Tanh(),
    Linear(32, 32),
    Tanh(),
    Linear(32, 1),
)

optimizer = SGD(model.parameters(), lr=0.05)

## Training

In jeder Epoche passiert Folgendes:

1. Wir mischen die Trainingsdaten.
2. Wir teilen sie in kleine Mini-Batches.
3. Für jedes Batch:
   - Vorwärtslauf
   - Verlust berechnen
   - Rückwärtslauf (`backward`)
   - Parameterupdate

Zusätzlich speichern wir Trainings- und Validierungsfehler.

In [ ]:
num_epochs = 2000
batch_size = 32

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    permutation = np.random.permutation(len(x_train))
    epoch_loss = 0.0

    for start in range(0, len(x_train), batch_size):
        batch_idx = permutation[start:start + batch_size]

        xb = Tensor(x_train[batch_idx])
        yb = Tensor(y_train[batch_idx])

        prediction = model(xb)
        loss = mse_loss(prediction, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.data.item()

    train_loss = epoch_loss / int(np.ceil(len(x_train) / batch_size))
    val_prediction = model(Tensor(x_val))
    val_loss = mse_loss(val_prediction, Tensor(y_val)).data.item()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 200 == 0:
        print(f"Epoche {epoch:4d}: Trainingsfehler = {train_loss:.6f}, Validierungsfehler = {val_loss:.6f}")

## Lernkurven

Wenn alles korrekt implementiert ist, sollten beide Fehler im Verlauf deutlich kleiner werden.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="Training")
plt.plot(val_losses, label="Validierung")
plt.yscale("log")
plt.xlabel("Epoche")
plt.ylabel("MSE")
plt.title("Lernkurven")
plt.legend()
plt.show()

## Ergebnis: Wie gut approximiert das Netz die Funktion?

Jetzt vergleichen wir die wahre Funktion \( \sin(x) \) mit der Vorhersage des trainierten Netzes.

In [ ]:
x_plot = np.linspace(-np.pi, np.pi, 400).reshape(-1, 1)
y_true = np.sin(x_plot)
y_pred = model(Tensor(x_plot)).data

plt.figure(figsize=(7, 4))
plt.plot(x_plot, y_true, label="sin(x)", linewidth=2)
plt.plot(x_plot, y_pred, "--", label="Netzvorhersage", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Approximation von sin(x)")
plt.legend()
plt.show()

## Zusammenfassung

In diesem Notebook haben wir

- ein kleines neuronales Netz **mit NumPy** gebaut,
- eine einfache Form der **automatischen Differenzierung** implementiert,
- die Gradienten mit **finiten Differenzen** geprüft,
- und das Netz erfolgreich auf einer Regressionsaufgabe trainiert.

### Was wichtig ist

Die entscheidende Idee ist nicht das einzelne Python-Kommando, sondern der Zusammenhang:

1. Ein Vorwärtslauf erzeugt einen Fehlerwert.
2. Dieser Fehlerwert hängt über viele Operationen von den Parametern ab.
3. Mit der Kettenregel kann man die Ableitungen systematisch rückwärts berechnen.
4. Diese Ableitungen sagen uns, wie wir die Parameter ändern müssen.

### Nächste sinnvolle Schritte

Wenn Sie dieses Notebook verstanden haben, können Sie als Nächstes zum Beispiel

- weitere Aktivierungsfunktionen ergänzen,
- Adam statt SGD implementieren,
- Klassifikation statt Regression betrachten,
- oder dieselbe Aufgabe mit PyTorch vergleichen.